# 2 · The gating motion, and why symmetry matters

PIEZO1 opens when membrane tension flattens its dome. This notebook builds an
**elastic network model** from a single closed structure and asks whether the
motion the protein makes most cheaply is the motion it actually makes.

It is a real test, because the answer is checkable: two deposited structures
give the observed transition, and the model never sees the second one.

In [ ]:
import numpy as np

from piezo1.config import STRUCTURE_DIR
from piezo1.core.structure import Structure
from piezo1.physics.anm import ANM
from piezo1.structure.protomers import protomer_blocks

curved = Structure.from_file(STRUCTURE_DIR / "7WLT.cif")
flat = Structure.from_file(STRUCTURE_DIR / "7WLU.cif")

_, curved_res = protomer_blocks(curved)
_, flat_res = protomer_blocks(flat)
common = np.array(sorted(set(curved_res.tolist()) & set(flat_res.tolist())))
print(f"{len(common)} residues resolved in all six protomers")

## Three things to get right before any of this means anything

Comparing two structures is where most of the mistakes live, and all three of
these produce a plausible wrong number rather than an error.

1. **Compare like with like.** The two entries do not resolve the same
   residues, so both are resampled onto the set they share. Skip this and you
   are comparing different molecules.
2. **Do not trust chain labels.** They do not encode rotational order.
   Overlaying 7WLU on 7WLT by label gives 90.7 Å; searching the protomer
   correspondence gives 12.3 Å. `match_protomers` searches.
3. **Remove rigid-body motion.** A structure that has merely been *moved* would
   otherwise look like a huge conformational change. Kabsch superposition
   takes it out, leaving only the shape change.

In [ ]:
from piezo1.structure.superpose import kabsch, match_protomers


def resample(structure, residues):
    """The same residues, in the same order, from each protomer."""
    out = []
    for chain in structure.chains:
        mask = structure.mask_ca() & (structure.chain == chain)
        if mask.sum() < 300:
            continue
        index = {int(r): i for i, r in enumerate(structure.res_seq[mask])}
        xyz = structure.xyz[mask]
        out.append(np.array([xyz[index[r]] for r in residues], dtype=float))
    return out[:3]


curved_blocks = resample(curved, common)
flat_blocks = resample(flat, common)

match = match_protomers(curved_blocks, flat_blocks)
flat_blocks = [flat_blocks[i] for i in match.order]
print("protomer order found:", match.order, " RMSD", round(match.rmsd, 1), "A")

rotation, translation, centroid = kabsch(np.vstack(flat_blocks),
                                         np.vstack(curved_blocks))
fitted = (np.vstack(flat_blocks) - centroid) @ rotation.T + translation
displacement = (fitted - np.vstack(curved_blocks)).ravel()

rmsd = float(np.sqrt((displacement ** 2).reshape(-1, 3).sum(1).mean()))
print(f"shape change after removing rigid motion: {rmsd:.1f} A RMSD")

## Build the elastic network, and label every mode by symmetry

The channel is a trimer, so every normal mode transforms as one of the
irreducible representations of C3: **A** (symmetric under 120° rotation) or
**E** (a degenerate pair that is not).

This is not bookkeeping. Membrane tension is isotropic, so it is itself
three-fold symmetric — and a symmetric perturbation cannot drive an
antisymmetric motion at first order. **Only A modes can be gating
coordinates.** E modes are forbidden by symmetry, whatever their frequency.

Note the model is built on the **curved** structure alone. It never sees the
flattened one.

In [ ]:
anm = ANM.from_trimer(curved_blocks, cutoff=15.0,
                      spring="inverse_square").build()
modes = anm.calc_modes(n_modes=30)
anm.label_symmetry(modes)

print(f"{modes.n_modes} modes over {anm.n_sites * 3} degrees of freedom\n")
for i in range(8):
    print(f"  mode {i:2d}   symmetry {modes.symmetry[i]:2s}   "
          f"frequency {modes.frequencies[i]:.5f}")

## Compare with the observed transition

The overlap between a mode and the observed displacement is the cosine between
them: 1.0 is a perfect match, 0.0 is unrelated.

In [ ]:
overlaps = np.abs(np.asarray(modes.overlap(displacement), dtype=float))
order = np.argsort(-overlaps)

print("best modes by overlap with the observed change:")
for i in order[:5]:
    print(f"  mode {i:2d}  symmetry {modes.symmetry[i]:2s}  "
          f"overlap {overlaps[i]:.4f}")

## The result

A single symmetric mode captures most of the transition, and every
symmetry-forbidden mode scores essentially zero. The model does not merely fit
the change; it finds it through the channel the physics permits.

In [ ]:
a_modes = [i for i in range(modes.n_modes) if modes.symmetry[i] == "A"]
e_modes = [i for i in range(modes.n_modes) if modes.symmetry[i] == "E"]

best_a = max(overlaps[i] for i in a_modes)
best_e = max(overlaps[i] for i in e_modes)
share = sum(overlaps[i] ** 2 for i in a_modes) / float((overlaps ** 2).sum())

print(f"best A-mode overlap    : {best_a:.4f}")
print(f"best E-mode overlap    : {best_e:.4f}")
print(f"share of overlap in A  : {share:.2%}")
print(f"cumulative over {modes.n_modes} modes: "
      f"{modes.cumulative_overlap(displacement)[-1]:.4f}")

assert best_a > 0.6, "the symmetric mode should capture most of the transition"
assert best_e < 0.05, "a forbidden mode should score essentially zero"
assert share > 0.99, "the overlap should sit almost entirely in A"

## What this does *not* say

The overlap depends on the elastic-network cutoff: over 10–20 Å it ranges from
0.554 to 0.723. The qualitative result — one symmetric mode, forbidden modes at
zero — survives every cutoff. The third decimal place does not, and
`docs/SCIENCE.md` publishes the range rather than the point estimate.

An elastic network also says nothing about **energetics**. It tells you the
motion is cheap, not that tension is enough to drive it. That calculation is in
`piezo1.physics.dome` and `piezo1.physics.elastica`, and it is where the linear
Helfrich theory usually applied to PIEZO1 turns out to overestimate the
footprint energy by 3.65×.

In [ ]:
from piezo1.analysis.uncertainty import sensitivity

def best_overlap(cutoff):
    trial = ANM.from_trimer(curved_blocks, cutoff=cutoff,
                            spring="inverse_square").build()
    return float(abs(trial.calc_modes(n_modes=20).overlap(displacement)).max())

spread = sensitivity(best_overlap, [10.0, 13.0, 15.0, 18.0, 20.0],
                     knob="anm.cutoff", what="best mode overlap")
print(spread.summary())
print("\nThis is a SENSITIVITY range over a method choice.")
print("It is explicitly not a confidence interval, and the class refuses to")
print("call it one.")